[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohamedEshmawy/langchain-handbook-gemini/blob/main/08-langchain-retrieval-agent.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/MohamedEshmawy/langchain-handbook-gemini/blob/main/08-langchain-retrieval-agent.ipynb)

#### [LangChain Handbook](https://pinecone.io/learn/langchain)

# Retrieval Agents

We've seen in previous chapters how powerful [retrieval augmentation](https://www.pinecone.io/learn/langchain-retrieval-augmentation/) and [conversational agents](https://www.pinecone.io/learn/langchain-agents/) can be. They become even more impressive when we begin using them together.

Conversational agents can struggle with data freshness, knowledge about specific domains, or accessing internal documentation. By coupling agents with retrieval augmentation tools we no longer have these problems.

One the other side, using "naive" retrieval augmentation without the use of an agent means we will retrieve contexts with *every* query. Again, this isn't always ideal as not every query requires access to external knowledge.

Merging these methods gives us the best of both worlds. In this notebook we'll learn how to do this.

To begin, we must install the prerequisite libraries that we will be using in this notebook.

In [1]:
# Use %pip (not !pip) so packages install into THIS notebook's kernel,
# not some other Python on your system. Restart the kernel after this runs.
%pip install -qU     langchain-qdrant==0.2.1 qdrant-client     langchain==0.3.25     langchain-community==0.3.25     langchain-google-genai==2.0.10     sniffio anyio pyparsing     tiktoken     datasets

# --- OpenRouter alternative (for reference) ---
# %pip install -qU langchain-openai==0.3.22

Note: you may need to restart the kernel to use updated packages.


## Building the Knowledge Base

We start by constructing our knowledge base. We'll use a mostly prepared dataset called **S**tanford **Qu**estion-**A**nswering **D**ataset (SQuAD) hosted on Hugging Face *Datasets*. We download it like so:

In [2]:
from datasets import load_dataset

data = load_dataset('squad', split='train')
data

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 87599
})

The dataset does contain duplicate contexts, which we can remove like so:

In [3]:
data = data.to_pandas()
data.head()

,id,title,context,question,answers
0,5733be284776f41900661182,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",To whom did the Virgin Mary allegedly appear i...,"{'text': ['Saint Bernadette Soubirous'], 'answ..."
1,5733be284776f4190066117f,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is in front of the Notre Dame Main Building?,"{'text': ['a copper statue of Christ'], 'answe..."
2,5733be284776f41900661180,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",The Basilica of the Sacred heart at Notre Dame...,"{'text': ['the Main Building'], 'answer_start'..."
3,5733be284776f41900661181,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is the Grotto at Notre Dame?,{'text': ['a Marian place of prayer and reflec...
4,5733be284776f4190066117e,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What sits on top of the Main Building at Notre...,{'text': ['a golden statue of the Virgin Mary'...


In [4]:
data.drop_duplicates(subset='context', keep='first', inplace=True)

# SQuAD has ~87k rows / ~18k unique contexts - WAY too big to embed on the free tier.
# Cap the number of rows we actually embed. Increase for more data.
SAMPLE_SIZE = 100  # increase for more data
data = data.head(SAMPLE_SIZE)
data.head()

,id,title,context,question,answers
0,5733be284776f41900661182,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",To whom did the Virgin Mary allegedly appear i...,"{'text': ['Saint Bernadette Soubirous'], 'answ..."
5,5733bf84d058e614000b61be,University_of_Notre_Dame,"As at most other universities, Notre Dame's st...",When did the Scholastic Magazine of Notre dame...,"{'text': ['September 1876'], 'answer_start': [..."
10,5733bed24776f41900661188,University_of_Notre_Dame,The university is the major seat of the Congre...,Where is the headquarters of the Congregation ...,"{'text': ['Rome'], 'answer_start': [119]}"
15,5733a6424776f41900660f51,University_of_Notre_Dame,The College of Engineering was established in ...,How many BS level degrees are offered in the C...,"{'text': ['eight'], 'answer_start': [487]}"
20,5733a70c4776f41900660f64,University_of_Notre_Dame,All of Notre Dame's undergraduate students are...,What entity provides help with the management ...,"{'text': ['Learning Resource Center'], 'answer..."


### Initialize the Embedding Model and Vector DB

We'll be using Google's `gemini-embedding-001` model initialized via LangChain and the Qdrant vector DB. We start by initializing the embedding model, for this we need a [Google API key](https://aistudio.google.com/app/apikey).

In [5]:
import os
from getpass import getpass
from langchain_google_genai import GoogleGenerativeAIEmbeddings

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY") or getpass("Enter your Google API key: ")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# --- OpenRouter alternative (for reference) ---
# os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY") or getpass("Enter your OpenRouter API key: ")
# OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

embed = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY,
)

# --- OpenAI alternative (for reference) ---
# from langchain.embeddings.openai import OpenAIEmbeddings
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or getpass("Enter your OpenAI API key: ")
# embed = OpenAIEmbeddings(
#     model="text-embedding-ada-002",
#     openai_api_key=OPENAI_API_KEY,
# )

D:\Work\DEBI\Agents\.venv-langchain\Lib\site-packages\langchain_google_genai\chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


Now we create our vector DB to store our vectors. For this we connect to an existing Qdrant deployment. Qdrant here needs no API key — we just point the client at the deployment URL.

In [6]:
# Qdrant here needs no API key - we just point the client at the deployment URL.
QDRANT_URL = "http://109.199.118.73:6333"

Now we connect to Qdrant and create a collection to hold our vectors.

In [7]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

client = QdrantClient(url=QDRANT_URL)
collection_name = "handbook-08-retrieval-agent"

Creating a collection, we set `size` equal to the dimensionality of `gemini-embedding-001` (`3072`), and use a `distance` metric compatible with it (`COSINE`).

In [8]:
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE),  # gemini-embedding-001 dims
    )

# view collection info
client.get_collection(collection_name)

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=0, segments_count=2, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=3072, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent_unoptimized=None)

We should see that the new Qdrant collection has a `points_count` of `0`, as we haven't added any vectors yet.

## Indexing

We can perform the indexing task using the LangChain vector store object. We will do this in batches of `100` or more.

In [9]:
from langchain_qdrant import QdrantVectorStore
from tqdm.auto import tqdm
import uuid

# LangChain vector store backed by our Qdrant collection
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embed,
)

batch_size = 100

for i in tqdm(range(0, len(data), batch_size)):
    # get end of batch
    i_end = min(len(data), i+batch_size)
    batch = data.iloc[i:i_end]
    # metadata fields for each record
    metadatas = [{
        'title': record['title'],
        'text': record['context']
    } for j, record in batch.iterrows()]
    # the list of contexts / documents
    documents = list(batch['context'])
    # ids for each record - Qdrant requires uint or UUID, so derive a stable
    # UUID from each SQuAD string id (which is a hex hash, not a valid point id)
    ids = [str(uuid.uuid5(uuid.NAMESPACE_DNS, str(_id))) for _id in batch['id']]
    # add everything to qdrant (embeddings are computed by the vector store)
    vectorstore.add_texts(texts=documents, metadatas=metadatas, ids=ids)

  0%|          | 0/1 [00:00<?, ?it/s]

We've indexed everything, now we can check the number of vectors in our collection like so:

In [10]:
client.get_collection(collection_name)

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=100, segments_count=2, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=3072, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent_unoptimized=Non

## Creating a Vector Store and Querying

Now that we've built our index we can switch back over to LangChain. The `vectorstore` we created above is already wired to the same Qdrant collection, so we can reuse it directly. (If you were starting a fresh session, you could reconnect with `QdrantVectorStore(client=client, collection_name=collection_name, embedding=embed)`.)

In [11]:
from langchain_qdrant import QdrantVectorStore

# (Re)connect a LangChain vector store to our existing Qdrant collection.
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embed,
)

As in previous examples, we can use the `similarity_search` method to do a pure semantic search (without the generation component).

In [12]:
query = "when was the college of engineering in the University of Notre Dame established?"

vectorstore.similarity_search(
    query,  # our search query
    k=3  # return 3 most relevant docs
)

[Document(metadata={'title': 'University_of_Notre_Dame', 'text': 'The College of Engineering was established in 1920, however, early courses in civil and mechanical engineering were a part of the College of Science since the 1870s. Today the college, housed in the Fitzpatrick, Cushing, and Stinson-Remick Halls of Engineering, includes five departments of study – aerospace and mechanical engineering, chemical and biomolecular engineering, civil engineering and geological sciences, computer science and engineering, and electrical engineering – with eight B.S. degrees offered. Additionally, the college offers five-year dual degree programs with the Colleges of Arts and Letters and of Business awarding additional B.A. and Master of Business Administration (MBA) degrees, respectively.', '_id': 'c1346054-9d9a-5382-8eb6-4e2f9e0b1032', '_collection_name': 'handbook-08-retrieval-agent'}, page_content='The College of Engineering was established in 1920, however, early courses in civil and mechan

Looks like we're getting good results. Let's take a look at how we can begin integrating this into a conversational agent.

## Initializing the Conversational Agent

Our conversational agent needs a Chat LLM, conversational memory, and a `RetrievalQA` chain to initialize. We create these using:

In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains.conversation.memory import ConversationBufferWindowMemory
from langchain.chains import RetrievalQA

# chat completion llm
llm = ChatGoogleGenerativeAI(
    temperature=0,
    google_api_key=GOOGLE_API_KEY,
    model="gemini-2.5-flash",
)

# --- OpenRouter alternative (for reference) ---
# OpenRouter is OpenAI-compatible: use the ChatOpenAI class with a different base_url.
# Pick any ":free" model from https://openrouter.ai/models, e.g. "openai/gpt-oss-120b:free".
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(
#     model="openai/gpt-oss-120b:free",
#     temperature=0,
#     api_key=OPENROUTER_API_KEY,
#     base_url="https://openrouter.ai/api/v1",
# )

# conversational memory
conversational_memory = ConversationBufferWindowMemory(
    memory_key='chat_history',
    k=5,
    return_messages=True
)
# retrieval qa chain
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)

C:\Users\pc\AppData\Local\Temp\ipykernel_67888\674785876.py:24: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  conversational_memory = ConversationBufferWindowMemory(


Using these we can generate an answer using the `run` method:

In [14]:
qa.run(query)

C:\Users\pc\AppData\Local\Temp\ipykernel_67888\2828950282.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  qa.run(query)


'The College of Engineering at the University of Notre Dame was established in 1920.'

But this isn't yet ready for our conversational agent. For that we need to convert this retrieval chain into a tool. We do that like so:

In [15]:
from langchain.agents import Tool

tools = [
    Tool(
        name='Knowledge Base',
        func=qa.run,
        description=(
            'use this tool when answering general knowledge queries to get '
            'more information about the topic'
        )
    )
]

Now we can initialize the agent like so:

In [16]:
from langchain.agents import initialize_agent

agent = initialize_agent(
    agent='chat-conversational-react-description',
    tools=tools,
    llm=llm,
    verbose=True,
    max_iterations=3,
    early_stopping_method='generate',
    memory=conversational_memory
)

C:\Users\pc\AppData\Local\Temp\ipykernel_67888\640423283.py:3: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(


With that our retrieval augmented conversational agent is ready and we can begin using it.

### Using the Conversational Agent

To make queries we simply call the `agent` directly.

In [17]:
agent(query)

C:\Users\pc\AppData\Local\Temp\ipykernel_67888\4024130983.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  agent(query)




> Entering new AgentExecutor chain...


```json
{
    "action": "Knowledge Base",
    "action_input": "when was the college of engineering in the University of Notre Dame established?"
}
```


Observation: The College of Engineering was established in 1920.
Thought:

```json
{
    "action": "Final Answer",
    "action_input": "The College of Engineering at the University of Notre Dame was established in 1920, according to the information obtained."
}
```

> Finished chain.


{'input': 'when was the college of engineering in the University of Notre Dame established?',
 'chat_history': [],
 'output': 'The College of Engineering at the University of Notre Dame was established in 1920, according to the information obtained.'}

Looks great, now what if we ask it a non-general knowledge question?

In [18]:
agent("what is 2 * 7?")



> Entering new AgentExecutor chain...


```json
{
    "action": "Final Answer",
    "action_input": "2 * 7 is 14."
}
```

> Finished chain.


{'input': 'what is 2 * 7?',
 'chat_history': [HumanMessage(content='when was the college of engineering in the University of Notre Dame established?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The College of Engineering at the University of Notre Dame was established in 1920, according to the information obtained.', additional_kwargs={}, response_metadata={})],
 'output': '2 * 7 is 14.'}

Perfect, the agent is able to recognize that it doesn't need to refer to it's general knowledge tool for that question. Let's try some more questions.

In [19]:
agent("can you tell me some facts about the University of Notre Dame?")



> Entering new AgentExecutor chain...


```json
{
    "action": "Knowledge Base",
    "action_input": "University of Notre Dame facts"
}
```


Observation: Here are some facts about the University of Notre Dame:

*   **Full Name & Meaning:** Its full name is the University of Notre Dame du Lac, which means "Our Lady of the Lake" in French, referring to its patron saint, the Virgin Mary.
*   **Location:** It is a Catholic research university located adjacent to South Bend, Indiana, in the United States.
*   **Campus:** The main campus covers 1,250 acres in a suburban setting.
*   **Landmarks:** Recognizable landmarks include the Golden Dome, the "Word of Life" mural (commonly known as Touchdown Jesus), and the Basilica.
*   **Academics & Rankings:** It is a large, four-year, highly residential research university, consistently ranked among the top twenty universities in the U.S. and considered a major global university.
*   **Undergraduate Structure:** The undergraduate component is organized into four colleges (Arts and Letters, Science, Engineering, Business) and the Architecture School.
*   **Architecture School:** The Arc

```json
{
    "action": "Final Answer",
    "action_input": "The University of Notre Dame du Lac, meaning \"Our Lady of the Lake,\" is a Catholic research university located adjacent to South Bend, Indiana. Its main campus spans 1,250 acres and features landmarks like the Golden Dome and the \"Word of Life\" mural. It is consistently ranked among the top twenty universities in the U.S. and is known for its four undergraduate colleges, the Architecture School, and over 50 graduate programs. Over 80% of its undergraduates live on campus in one of 29 single-sex residence halls. The university has a strong alumni network of approximately 120,000. Its athletic teams, known as the Fighting Irish, are famous for their football program, which has accumulated eleven consensus national championships. Other teams have won 16 national championships. The Notre Dame Victory March is a widely recognized collegiate fight song. The university also has an Office of Sustainability, with dining services s

{'input': 'can you tell me some facts about the University of Notre Dame?',
 'chat_history': [HumanMessage(content='when was the college of engineering in the University of Notre Dame established?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The College of Engineering at the University of Notre Dame was established in 1920, according to the information obtained.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='what is 2 * 7?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='2 * 7 is 14.', additional_kwargs={}, response_metadata={})],
 'output': 'The University of Notre Dame du Lac, meaning "Our Lady of the Lake," is a Catholic research university located adjacent to South Bend, Indiana. Its main campus spans 1,250 acres and features landmarks like the Golden Dome and the "Word of Life" mural. It is consistently ranked among the top twenty universities in the U.S. and is known for its four undergraduate colleges, the Archite

In [20]:
agent("can you summarize these facts in two short sentences")



> Entering new AgentExecutor chain...


```json
{
    "action": "Final Answer",
    "action_input": "The University of Notre Dame is a highly-ranked Catholic research university in Indiana, known for its iconic campus, strong academic programs, and vibrant residential life. It is also widely recognized for its successful 'Fighting Irish' athletic teams, particularly football, and its commitment to sustainability and peace studies."
}
```

> Finished chain.


{'input': 'can you summarize these facts in two short sentences',
 'chat_history': [HumanMessage(content='when was the college of engineering in the University of Notre Dame established?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The College of Engineering at the University of Notre Dame was established in 1920, according to the information obtained.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='what is 2 * 7?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='2 * 7 is 14.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='can you tell me some facts about the University of Notre Dame?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The University of Notre Dame du Lac, meaning "Our Lady of the Lake," is a Catholic research university located adjacent to South Bend, Indiana. Its main campus spans 1,250 acres and features landmarks like the Golden Dome and the "Word of Life" mural. I

Looks great! We're also able to ask questions that refer to previous interactions in the conversation and the agent is able to refer to the conversation history to as a source of information.

That's all for this example of building a retrieval augmented conversational agent with Google Gemini and Qdrant and LangChain.

Once finished, we delete the Qdrant collection to save resources:

In [21]:
client.delete_collection(collection_name)

True

---